In [729]:
# !pip install -q google-genai underthesea transformers torch numpy qdrant-client uuid6
# !pip install qdrant-client sentence-transformers -q
# !pip install -q groq  pydantic

# code for chunking 

In [747]:
# str(uuid6.uuid7())

### import and config
import, danh sách can chi, check can chi trong văn bản (đầu trang hoặc đầu dòng (\n))  

Số lượng trang cho từng lần gọi api  

Các kí hiệu chọn để ngắt dòng (RecursiveCharacterTextSplitter )  

Số lượng word, min and max overlap, max context lenght của model embedding   


In [748]:
from __future__ import annotations

import abc
import json
import os
import pathlib
import re
import time
import urllib.request
import uuid6
from dataclasses import dataclass
from typing import Optional

import numpy as np
import torch
from pydantic import BaseModel
from transformers import AutoModel, AutoTokenizer
from underthesea import word_tokenize
from google.colab import userdata
from qdrant_client import QdrantClient, models
from qdrant_client.http.models import Distance, VectorParams, PointStruct

# 60-cycle sexagenary calendar names (Giáp Tý → Quý Hợi)
CAN_CHI_60: list[str] = [
    "Giáp Tý",  "Ất Sửu",  "Bính Dần", "Đinh Mão", "Mậu Thìn", "Kỷ Tỵ",
    "Canh Ngọ", "Tân Mùi", "Nhâm Thân","Quý Dậu",  "Giáp Tuất","Ất Hợi",
    "Bính Tý",  "Đinh Sửu","Mậu Dần",  "Kỷ Mão",   "Canh Thìn","Tân Tỵ",
    "Nhâm Ngọ", "Quý Mùi", "Giáp Thân","Ất Dậu",   "Bính Tuất","Đinh Hợi",
    "Mậu Tý",   "Kỷ Sửu",  "Canh Dần", "Tân Mão",  "Nhâm Thìn","Quý Tỵ",
    "Giáp Ngọ", "Ất Mùi",  "Bính Thân","Đinh Dậu", "Mậu Tuất", "Kỷ Hợi",
    "Canh Tý",  "Tân Sửu", "Nhâm Dần", "Quý Mão",  "Giáp Thìn","Ất Tỵ",
    "Bính Ngọ", "Đinh Mùi","Mậu Thân", "Kỷ Dậu",   "Canh Tuất","Tân Hợi",
    "Nhâm Tý",  "Quý Sửu", "Giáp Dần", "Ất Mão",   "Bính Thìn","Đinh Tỵ",
    "Mậu Ngọ",  "Kỷ Mùi",  "Canh Thân","Tân Dậu",  "Nhâm Tuất","Quý Hợi",
]


# Sort longest-first to prevent sub-string matches (e.g. "Giáp" before "Giáp Tý")
_CAN_CHI_PATTERN: str = "|".join(
    re.escape(cc) for cc in sorted(CAN_CHI_60, key=len, reverse=True)
)
print(f"in can chi nè : {_CAN_CHI_PATTERN}\n")

# check xem có trùng với danh sách 60 can chi không. và phải bắt đầu bằng \n
CAN_CHI_LINE_START_RE: re.Pattern = re.compile(
    r"^(?:\[\d{2}[ab]\]\s+)?(" + _CAN_CHI_PATTERN + r")(?=\s+|\,|[\[\(])",
    re.MULTILINE
)


# check xem dòng đầu tiên của trang có can chi không
# CAN_CHI_LINE_START_RE: re.Pattern = re.compile(
#     r"^(" + _CAN_CHI_PATTERN + r")(?=\s+|\,|[\[]|$)", re.MULTILINE
# )


# Danh sách tên các tháng trong năm (Tháng 1 → Tháng 12)
MONTHS_LIST: list[str] = [
    "Tháng 1", "Tháng 2", "Tháng 3", "Tháng 4", "Tháng 5", "Tháng 6",
    "Tháng 7", "Tháng 8", "Tháng 9", "Tháng 10", "Tháng 11", "Tháng 12"
]

# Tạo Pattern mẫu dạng: (Tháng 1|Tháng 2|...|Tháng 12)
_MONTHS_PATTERN: str = "|".join(MONTHS_LIST)

# Regex nhận diện Tháng ở đầu dòng (hỗ trợ cả trường hợp có mã [54a] hoặc không)
# MONTHS_LINE_START_RE: re.Pattern = re.compile(
#     r"^(?:\[\d{2}[ab]\]\s+)?(" + _MONTHS_PATTERN + r")(?=\s*|\,|[\[\(\.]|$)",
#     re.MULTILINE
# )


# CAN_CHI_LINE_START_RE: re.Pattern = re.compile(
#     r"^(?:\[\d{2}[ab]\]\s+)?(?:" + _CAN_CHI_PATTERN + r"|" + _MONTHS_PATTERN + r")(?=\s*|\,|[\[\(\.]|$)",
#     re.MULTILINE
# )


# Tìm thẻ và trang trong văn bản [1], [2], …  , [PAGE:12]
FOOTNOTE_REF_RE: re.Pattern = re.compile(r"\[(\d{1,2})\]")
PAGE_MARKER_RE: re.Pattern = re.compile(r"\n\s*\[\[\[PAGE:(\d+)\]\]\]\s*\n")


# Separator priority list for word-aware text splitting
SEPARATORS: list[str] = ["\n\n", "\n", ".", "?", "!", ";", ":", ",", " ", ""]


# Sliding-window page counts per book
BOOK_WINDOW_SIZES: dict[str, int] = {
    "Đại Việt Sử Ký Toàn Thư":                5,
    "Khâm Định Việt Sử Thông Giám Cương Mục":   3,
    "Việt Sử Toàn Thư":                        3,
    "Vương Triều Trần":                         4,
}
DEFAULT_WINDOW_SIZE: int = 5


# Mini-chunk & merge sizing
MINI_CHUNK_WORDS: int  = 35     # ≈ 55-70 BPE tokens for classical Vietnamese
OVERLAP_TOK_MIN:  int  = 20
OVERLAP_TOK_MAX:  int  = 30
MAX_EMBED_TOKENS: int  = 256    # hard limit for BKAI bi-encoder


# Model IDs
EMBED_MODEL_ID:  str = "bkai-foundation-models/vietnamese-bi-encoder"
# GEMINI_MODEL_ID: str = "gemini-2.5-flash"
GEMINI_MODEL_ID: str = "gemini-3.1-flash-lite"

GROQ_MODEL_LLAMA_70B = "llama-3.3-70b-versatile"
GROQ_MODEL_LLAMA_8B  = "llama-3.1-8b-instant"
GROQ_MODEL_QWEN      = "qwen/qwen3-32b"



# Lấy cấu hình bảo mật từ Secrets của Colab
QDRANT_URL = userdata.get('QDRANT_URL')
API_KEY = userdata.get('QDRANT_API_KEY')

client = QdrantClient(url=QDRANT_URL, api_key=API_KEY)

COLLECTION_NAME = "history_social"
VECTOR_SIZE = 768  # Expected vector size from the embedding model

# Check if collection exists and its configuration matches
if client.collection_exists(collection_name=COLLECTION_NAME):
    collection_info = client.get_collection(collection_name=COLLECTION_NAME).config
    current_vector_size = collection_info.params.vectors.size

    if current_vector_size != VECTOR_SIZE:
        print(f"Lỗi: Collection '{COLLECTION_NAME}' đã tồn tại với kích thước vector {current_vector_size}, "
              f"nhưng đang mong đợi {VECTOR_SIZE}. Đang xóa và tạo lại collection...")
        client.delete_collection(collection_name=COLLECTION_NAME)
        client.create_collection(
            collection_name=COLLECTION_NAME,
            vectors_config=VectorParams(size=VECTOR_SIZE, distance=Distance.COSINE)
        )
        print(f"Đã tạo lại collection: {COLLECTION_NAME} với kích thước vector {VECTOR_SIZE}.")
    else:
        print(f"Collection '{COLLECTION_NAME}' đã tồn tại và kích thước vector khớp ({VECTOR_SIZE}).")
else:
    # Collection does not exist, create it
    client.create_collection(
        collection_name=COLLECTION_NAME,
        vectors_config=VectorParams(size=VECTOR_SIZE, distance=Distance.COSINE)
    )
    print(f"Đã khởi tạo collection mới: {COLLECTION_NAME} với kích thước vector {VECTOR_SIZE}.")

in can chi nè : Nhâm\ Thân|Giáp\ Tuất|Canh\ Thìn|Giáp\ Thân|Bính\ Tuất|Nhâm\ Thìn|Bính\ Thân|Giáp\ Thìn|Canh\ Tuất|Bính\ Thìn|Canh\ Thân|Nhâm\ Tuất|Bính\ Dần|Đinh\ Mão|Mậu\ Thìn|Canh\ Ngọ|Đinh\ Sửu|Nhâm\ Ngọ|Đinh\ Hợi|Canh\ Dần|Giáp\ Ngọ|Đinh\ Dậu|Mậu\ Tuất|Nhâm\ Dần|Bính\ Ngọ|Đinh\ Mùi|Mậu\ Thân|Giáp\ Dần|Giáp\ Tý|Tân\ Mùi|Quý\ Dậu|Bính\ Tý|Mậu\ Dần|Quý\ Mùi|Tân\ Mão|Canh\ Tý|Tân\ Sửu|Quý\ Mão|Tân\ Hợi|Nhâm\ Tý|Quý\ Sửu|Đinh\ Tỵ|Mậu\ Ngọ|Tân\ Dậu|Quý\ Hợi|Ất\ Sửu|Ất\ Hợi|Kỷ\ Mão|Tân\ Tỵ|Ất\ Dậu|Mậu\ Tý|Kỷ\ Sửu|Quý\ Tỵ|Ất\ Mùi|Kỷ\ Hợi|Kỷ\ Dậu|Ất\ Mão|Kỷ\ Mùi|Kỷ\ Tỵ|Ất\ Tỵ

Collection 'history_social' đã tồn tại và kích thước vector khớp (768).


### Data Structures
$$\text{Văn bản thô (chứa [PAGE:N])} \longrightarrow \text{PageSpan} \longrightarrow \text{MiniChunk} \longrightarrow \text{FinalChunk (Lưu Database)}$$

In [749]:
@dataclass
class PageSpan:
    """One [PAGE:N] block from source text, with the marker stripped."""
    page_no:      int   # -1 if text precedes the first marker
    start_offset: int   # char offset in the reconstructed marker-free text
    end_offset:   int
    text:         str


@dataclass
class MiniChunk:
    """Small ~35-word fragment before LLM semantic merging."""
    idx:           int
    text:          str


    start_offset:  int   # absolute offset in clean-text coordinate space
    end_offset:    int
    pages:         list[int]
    footnote_refs: list[str]   # ordered, deduplicated


@dataclass
class FinalChunk:
    """Production-ready chunk post-embedding."""
    chunk_id:       str
    book_name:      str
    pages:          list[int]
    raw_text:       str          # exact source text, never mutated
    segmented_text: str          # underthesea-segmented version (for embedding)
    footnote_refs:  list[str]
    footnotes:      dict[str, str]
    token_count:    int
    # vector:         list[float]


###  PAGE PARSER
Tách trang: Lưu vị trí đầu trang, cuối trang, nội dung trang  
Trả về văn bản sạch không có số trang  
Trả về số trang khi người dùng đưa cho một PageSpan  


In [750]:



def parse_page_spans(full_text: str) -> list[PageSpan]:
    # tìm thẻ page
    markers = list(PAGE_MARKER_RE.finditer(full_text))
    # print(f"tìm thẻ page: {markers}\n")

    spans:   list[PageSpan] = []
    cursor = 0   # con trỏ vị trí (vị trí tiếp theo của trang)

 # Nếu không thấy thì là trang -1 và toàn bộ văn bản đều ở 1 trang
    if not markers:
        spans.append(PageSpan(-1, 0, len(full_text), full_text))
        # print(f"[PARSER] No [PAGE:N] markers found — "
        #       f"entire text treated as page -1 ({len(full_text):,} chars)")
        return spans

    # xử lí văn bản trước thẻ PAGE
    pre = full_text[: markers[0].start()]
    if pre:
        spans.append(PageSpan(-1, 0, len(pre), pre))
        cursor = len(pre)   # chiều dài của văn bản trước PAGE
        # print(f"[PARSER] Pre-marker text: {len(pre):,} chars → page_no=-1")

# Vòng lặp xử lý từng trang (i số page tìm được còn m là đối tượng )
    for i, m in enumerate(markers):
        page_no   = int(m.group(1))    # lấy phần sau (chứa số trang )
        raw_start = m.end()
        raw_end   = markers[i + 1].start() if i + 1 < len(markers) else len(full_text)
        page_text = full_text[raw_start:raw_end]   # văn bản trong 1 trang

        span_start = cursor
        span_end   = cursor + len(page_text)
        spans.append(PageSpan(page_no, span_start, span_end, page_text))    # lưu các thông tin của trang
        cursor = span_end
        # print(f"Nội dung của một trang {page_text} \n\n\n\n\n\n")
    # print(f"[PARSER] {len(spans)} PageSpans | "
    #       f"pages {spans[0].page_no} → {spans[-1].page_no} | "
    #       f"clean-text total {cursor:,} chars")
    return spans



# trả về toàn bộ văn bản sạch không có page
def build_clean_text(spans: list[PageSpan]) -> str:
    return "".join(s.text for s in spans)




# Tìm xem đoạn từ ký tự [start] đến [end] đang nằm ở (những) trang sách số mấy? trả về [4, 5]
def map_pages_to_slice(
    spans: list[PageSpan], start: int, end: int
) -> list[int]:
    result: list[int] = []
    for s in spans:
        if s.end_offset   <= start:
            continue
        if s.start_offset >= end:
            break
        if s.page_no != -1:
            result.append(s.page_no)
    return sorted(set(result))


### Cắt theo can chi (window text)
Cắt theo trang mặc định sau đó mới duyệt ngược lại cắt theo can chi

In [751]:
@dataclass
class WindowResult:
    text: str
    pages: list[int]
    global_start: int
    global_end: int


def _get_window_size(book_name: str) -> int:
    for key, size in BOOK_WINDOW_SIZES.items():
        if key.lower() in book_name.lower() or book_name.lower() in key.lower():
            return size
    return DEFAULT_WINDOW_SIZE


def create_dynamic_can_chi_windows(
    spans: list[PageSpan], book_name: str
) -> list[WindowResult]:

    N = _get_window_size(book_name)
    print(f"\n[WINDOWS] book='{book_name}' | window_size={N} pages")

    windows: list[WindowResult] = []
    remainder_text: str = ""
    remainder_origin: int = 0

    real_spans = [s for s in spans if s.page_no != -1]
    if not real_spans:
        return []

    idx = 0
    total = len(spans)

    while idx < total or remainder_text.strip():

        batch = spans[idx: idx + N]
        is_last = idx + len(batch) >= total

        if not batch and not remainder_text.strip():
            break

        batch_text = "".join(s.text for s in batch) if batch else ""

        if remainder_text:
            pool_text = remainder_text + batch_text
            pool_start = remainder_origin
        else:
            pool_text = batch_text
            pool_start = batch[0].start_offset if batch else remainder_origin

        if not pool_text.strip():
            idx += len(batch)
            continue

        if is_last:
            cut_local = len(pool_text)

        else:
            last_page_text = batch[-1].text if batch else ""
            last_page_start_in_pool = len(pool_text) - len(last_page_text)

            cut_local = -1

            matches = list(CAN_CHI_LINE_START_RE.finditer(last_page_text))
            if matches:
                cut_local = last_page_start_in_pool + matches[-1].start()

            if cut_local <= 0:
                para = last_page_text.rfind("\n\n")
                if para != -1:
                    cut_local = last_page_start_in_pool + para + 2

            if cut_local <= 0:
                nl = last_page_text.rfind("\n")
                if nl != -1:
                    cut_local = last_page_start_in_pool + nl + 1

            if cut_local <= 0:
                dot = last_page_text.rfind(".")
                if dot != -1:
                    cut_local = last_page_start_in_pool + dot + 1

            if cut_local <= 0:
                cut_local = len(pool_text)

        window_text = pool_text[:cut_local]
        window_len = len(window_text)

        if (
            not is_last
            and window_len < 6000
            and (idx + len(batch)) < total
        ):
            extra_batch = spans[idx + len(batch): idx + len(batch) + 1]

            if extra_batch:
                extra_text = "".join(s.text for s in extra_batch)

                pool_text += extra_text

                # Tính lại điểm cắt dựa trên trang cuối mới
                last_page_text = extra_batch[-1].text
                last_page_start_in_pool = len(pool_text) - len(last_page_text)

                cut_local = -1

                matches = list(CAN_CHI_LINE_START_RE.finditer(last_page_text))

                if matches:
                    cut_local = last_page_start_in_pool + matches[-1].start()

                if cut_local <= 0:
                    para = last_page_text.rfind("\n\n")
                    if para != -1:
                        cut_local = last_page_start_in_pool + para + 2

                if cut_local <= 0:
                    nl = last_page_text.rfind("\n")
                    if nl != -1:
                        cut_local = last_page_start_in_pool + nl + 1

                if cut_local <= 0:
                    dot = last_page_text.rfind(".")
                    if dot != -1:
                        cut_local = last_page_start_in_pool + dot + 1

                if cut_local <= 0:
                    cut_local = len(pool_text)

                window_text = pool_text[:cut_local]
                window_len = len(window_text)

                batch.extend(extra_batch)



        window_start = pool_start
        window_end = pool_start + cut_local

        remainder_text = pool_text[cut_local:]
        remainder_origin = pool_start + cut_local

        idx += len(batch)

        if not window_text.strip():
            if is_last:
                break
            continue

        pages = map_pages_to_slice(spans, window_start, window_end)

        windows.append(WindowResult(
            text=window_text,
            pages=pages,
            global_start=window_start,
            global_end=window_end,
        ))

        if is_last:
            break

    return windows

In [752]:
# def run_pipeline(
#     input_json:         dict,
#     backends:           list[LLMBackend],
#     batch_orchestrator: Optional[GeminiBatchOrchestrator] = None,
# ) -> list[dict]:

#     book_name          = input_json["book_name"]
#     full_text          = input_json["full_text"]
#     page_footnotes_map = input_json["page_footnotes_map"]

# # bóc tách trang
#     spans = parse_page_spans(full_text)

# # chia sách thành các trang cho các lần gọi api
#     windows = create_dynamic_can_chi_windows(spans, book_name)

#     total_windows: int        = len(windows)

#     for wi, window in enumerate(windows):
#         print(f"\n{'─' * 72}")
#         pg_str = str(window.pages[:6]) + ("…" if len(window.pages) > 6 else "")
#         print(f"  WINDOW {wi + 1:>4}/{total_windows} | pages={pg_str} | chars={len(window.text):,} \n\n\n")

### Cấu trúc cắt
Cấu hình các thành phần có thể cắt tùy vào từng loại sách  

Cắt một chuỗi văn bản dài thành các đoạn nhỏ hơn


In [753]:


def detect_structural_boundaries(text: str, book_name: str) -> list[int]:
    bk = book_name
    offsets: set[int] = {0, len(text)}

    if "Đại Việt Sử Ký Toàn Thư" in bk:
        # Can-Chi names at the start of a line
        for m in CAN_CHI_LINE_START_RE.finditer(text):
            offsets.add(m.start())

    elif "Khâm Định Việt Sử Thông Giám Cương Mục" in bk:
        # Lines that begin with "Năm <Can-Chi>"
        pat = re.compile(
            r"^Năm\s+(?:" + _CAN_CHI_PATTERN + r")(?=\s|$)", re.MULTILINE
        )
        for m in pat.finditer(text):
            offsets.add(m.start())

    elif "Việt Sử Toàn Thư" in bk:
        # Numbered section markers: "12 - " at line start
        for m in re.finditer(r"^\d+\s*-\s*", text, re.MULTILINE):
            offsets.add(m.start())
        # Paragraph breaks (soft boundaries)
        for m in re.finditer(r"\n{1,}", text):
            offsets.add(m.end())

    elif "Vương Triều Trần" in bk:
        # Soft paragraph breaks only
        for m in re.finditer(r"\n{1,}", text):
            offsets.add(m.end())

    else:
        for m in re.finditer(r"\n{1,}", text):
            offsets.add(m.end())

    result = sorted(offsets)
    print(f"      [STRUCT] {len(result)} structural boundaries | book='{book_name[:35]}'")
    return result



def split_text_by_words(
    text: str, max_words: int, separators: list[str]
) -> list[tuple[int, int, str]]:
    if not text.strip():
        return []
    text = re.sub(r'(?<=[a-zà-ỹ])[\n\;](?=[a-zà-ỹ])', ' ', text)
    if len(text.split()) <= max_words:
        return [(0, len(text), text)]

    results: list[tuple[int, int, str]] = []
    cursor = 0

    while cursor < len(text):
        remaining = text[cursor:]
        if not remaining.strip():
            break

        if len(remaining.split()) <= max_words:
            results.append((cursor, len(text), remaining))
            break

        word_count = 0
        in_word    = False
        approx_end = len(remaining)
# đếm số lượng từ
        for ci, ch in enumerate(remaining):
            is_ws = ch in (" ", "\n", "\t", "\r")
            if is_ws:
                in_word = False
            elif not in_word:
                in_word = True
                word_count += 1
                if word_count > max_words:
                    approx_end = ci
                    break
        # print(f"Số lượng từ: {in_word} \n\n\n")
        # ── Walk backward to find the best separator at or before approx_end ─
        cut_local: int = -1
        for sep in separators:
            if sep == "":
                # Empty string = hard-cut at approx_end (last resort)
                cut_local = approx_end
                break
            search_bound = min(approx_end + len(sep), len(remaining))
            idx = remaining.rfind(sep, 0, search_bound)
            if idx != -1 and (idx + len(sep)) > 0:
                cut_local = idx + len(sep)
                break

        if cut_local <= 0:
            cut_local = approx_end
        if cut_local <= 0 or cut_local > len(remaining):
            cut_local = len(remaining)

        frag = remaining[:cut_local]   # exact source slice — no mutation
        if frag.strip():
            results.append((cursor, cursor + cut_local, frag))

        cursor += cut_local

    return results

### Mini Chunk Creation (rule-based pre-segmentation)
Từ các trang sách tách tiếp thành các thành phần bỏ hơn  
Từ các thành phần trên tách tiếp thành các chunk có kích thước khoản 35 word

In [754]:


def create_mini_chunks(
    window: WindowResult, spans: list[PageSpan], book_name: str
) -> list[MiniChunk]:

    text       = window.text
    # print(f"window text\n {text} \n\n\n")
    boundaries = detect_structural_boundaries(text, book_name)
    # print(f"boundaries\n {boundaries} \n\n\n")

    mini_chunks: list[MiniChunk] = []
    idx = 0

    for bi in range(len(boundaries) - 1):
        seg_start = boundaries[bi]
        seg_end   = boundaries[bi + 1]
        seg_text  = text[seg_start:seg_end]   # exact source slice
        # print(f"các phần nhỏ trong trang \n {seg_text} \n\n\n")
        if not seg_text.strip():
            continue

        frags = split_text_by_words(seg_text, MINI_CHUNK_WORDS, SEPARATORS)

        for (local_s, local_e, frag_text) in frags:
            if not frag_text.strip():
                continue
            # print(f"{frag_text} \n\n\n")
            # Absolute offsets in the global clean-text coordinate space
            abs_start = window.global_start + seg_start + local_s
            abs_end   = window.global_start + seg_start + local_e

            # Page mapping
            pages = map_pages_to_slice(spans, abs_start, abs_end)
            # print(f"map_pages_to_slice: \n {pages}")
            if not pages:
                pages = list(window.pages)   # fallback: window-level pages
                # print("méo hiểu kiểu gì \n\n\n")

            # Footnote refs: dict.fromkeys preserves insertion order & deduplicates
            refs = list(dict.fromkeys(FOOTNOTE_REF_RE.findall(frag_text)))
            # print(f"danh sách footer\n {refs} \n\n\n")
            mini_chunks.append(MiniChunk(
                idx=idx,
                text=frag_text,
                start_offset=abs_start,
                end_offset=abs_end,
                pages=pages,
                footnote_refs=refs,
            ))
            idx += 1

    print(f"    [MINI-CHUNK] window chars={len(text):,} → {len(mini_chunks)} mini-chunks")
    return mini_chunks

### call llm



#### promt và gán nhãn cho input,  output

Luồng chạy: tạo form cho các minichunk sau đó sử dụng llm. llm sử dungj

In [755]:
class SplitResponse(BaseModel):
    """
    Structured-output schema enforced on every LLM call (batch and sync).
    Guarantees the model can ONLY return a valid JSON object of this shape:
        {"split_after_indices": [3, 7, 12]}
    """
    split_after_indices: list[int]


_SYSTEM_PROMPT: str = """\
You are an expert semantic chunk-boundary detector specializing in Vietnamese historical chronicles, royal annals (Đại Việt Sử Ký Toàn Thư, Khâm Định Việt Sử...), and academic texts.

YOUR TASK:
Analyze a sequence of pre-split historical text fragments (mini-chunks) and determine ONLY the indices AFTER which a semantic split must occur to form optimal macro-chunks for a 256-token embedding model.

STRICT OPERATIONAL RULES:
1. NEVER rewrite, summarize, paraphrase, translate, or correct the source text.
2. NEVER output any words, phrases, or sentences from the text.
3. OUTPUT ONLY a single valid JSON object. No markdown blocks (```json), no conversational filler, no explanations.
4. Output ONLY a single valid JSON object on a single line, with absolutely no whitespaces, newlines, or markdown fences: {"split_after_indices":[3,7,12]}

VIETNAMESE HISTORICAL SEMANTIC SPLIT CRITERIA:
Insert a split index when a clear shift in historical context occurs, specifically:
- A change in Dynasty (Triều đại), Reign Era/Niên hiệu (e.g., Hồng Đức, Vĩnh Hựu).
- A change in the Historical Year or Can-Chi cycle (e.g., Canh Tý, Nhâm Thìn).
- A transition to a new historical figure, king, general, or envoy.
- A shift in geographic location, military campaigns, or diplomatic missions.
- A structural transition: Narrative text (Tự sự) <-> Royal Decree/Edict (Chiếu/Chỉ) <-> Commentary/Critique (Lời phê/Bình luận của sử gia).

ANTI-OVERFRAGMENTATION RULE:
- Do NOT split if a historical event, military battle, or single continuous dialogue spans across mini-chunks, UNLESS forced by the embedding constraints below.

EMBEDDING & TOKEN SIZE CONSTRAINTS (CRITICAL):
Each mini-chunk contains roughly 20-35 Vietnamese words (~35-55 tokens). Our embedding model has a HARD LIMIT of 256 tokens.
- IDEAL MACRO-CHUNK SIZE: 2 to 3 mini-chunks (~70 - 150 tokens).
- MAXIMUM CAPACITY: 4 mini-chunks (~160 - 220 tokens).
- FORCED SPLIT RULE: You MUST force a split at the 4th mini-chunk index, even if the historical narrative is in the middle of a sentence or event, to prevent token truncation downstream.

OUTPUT FORMAT:
Return EXACTLY a JSON object with a single key "split_after_indices" containing an array of 0-based local indices.
If the entire sequence forms one coherent unit under 4 mini-chunks, return an empty array.
"""

_USER_TEMPLATE: str = """\
The following {n} tagged mini-chunks are a sequential sub-batch from a Vietnamese historical text.
Review the text content and indices carefully. Output the split indices according to the system instructions.

[START OF MINI-CHUNKS]
{tagged}
[END OF MINI-CHUNKS]

Your JSON response:"""


# form cho các minichunk trước khi gửi cho llm
def _format_tagged_chunks(chunks: list[MiniChunk]) -> str:
    return "\n".join(
        f"<start_chunk_{mc.idx}>{mc.text}<end_chunk_{mc.idx}>"
        for mc in chunks
    )

#  chuẩn hóa vị trí của các chunk đều bắt đầu từ 0 khi gửi cho llm
def _format_tagged_chunks_zero_indexed(chunks: list[MiniChunk]) -> str:
    """
    Format for BATCH sub-batches — always re-indexes tags from 0.
    Ensures the LLM sees indices 0..N-1 regardless of mc.idx values,
    so _parse_split_response can apply the sub-batch offset cleanly.
    """
    return "\n".join(
        f"<start_chunk_{i}>{mc.text}<end_chunk_{i}>"
        for i, mc in enumerate(chunks)
    )

# Nhận câu trả lời của llm trả về mảng int chứa các vị trí cần cắt
def _parse_split_response(raw: str, n_chunks: int) -> list[int]:
    clean = re.sub(r"^```[a-z]*\n?", "", raw.strip())
    clean = re.sub(r"\n?```$", "", clean.strip())
    clean = re.sub(r"<think>.*?</think>", "", clean, flags=re.DOTALL).strip()

    parsed  = json.loads(clean)
    indices = parsed.get("split_after_indices", [])

    max_valid = n_chunks - 2
    # sort dùng để sắp xếp lại thứ tự nếu llm có trả về sai vị trí còn set dùng để loại các số trùng nhau
    return sorted(
        set(
            int(i) for i in indices
            if isinstance(i, (int, float)) and 0 <= int(i) <= max_valid
        )
    )


#### chọn llm

In [756]:
# Lớp abstract cho gọi các llm
class LLMBackend(abc.ABC):
    @property
    @abc.abstractmethod
    def label(self) -> str:
        # in ra tên model hay nội dung nào đó để log
        pass

    @abc.abstractmethod # nhận system promt và user promt rồi trả về chuỗi nếu lỗi trả về lỗi
    def call(self, system: str, user: str) -> str:
        pass



class GroqBackend(LLMBackend):

    def __init__(self, api_key: str, model_id: str = GROQ_MODEL_LLAMA_70B):
        from groq import Groq
        self._client = Groq(api_key=api_key)
        self._model  = model_id

    @property
    def label(self) -> str:
        return f"groq/{self._model}"

    def call(self, system: str, user: str) -> str:
        completion = self._client.chat.completions.create(
            model=self._model,
            messages=[
                {"role": "system", "content": system},
                {"role": "user",   "content": user},
            ],
            temperature=0.0,
            max_tokens=2048,
        )
        return completion.choices[0].message.content


class GeminiBackend(LLMBackend):

    def __init__(self, api_key: Optional[str] = None,
                 model_id: str = GEMINI_MODEL_ID):
        from google import genai
        from google.genai import types as _gtypes
        self._types  = _gtypes
        self._client = genai.Client(api_key=api_key) if api_key else genai.Client()
        self._model  = model_id

    @property
    def label(self) -> str:
        return f"gemini-sync/{self._model}"

    def call(self, system: str, user: str) -> str:
        resp = self._client.models.generate_content(
            model=self._model,
            contents=user,
            config=self._types.GenerateContentConfig(
                system_instruction=system,
                temperature=0.0,
                max_output_tokens=4096,
            ),
        )
        return resp.text


####  Gọi LLM để gộp các minichunk theo ngữ nghĩa

In [757]:

# hàm này dùng để chạy đồng bộ - Synchronous
# Chạy luôn từng cửa sổ văn bản
def llm_semantic_split(
    mini_chunks: list[MiniChunk],
    backends:    list[LLMBackend],
    max_retries: int = 3,
) -> list[int]:

    if len(mini_chunks) < 3:
        print(f"    [LLM-SYNC] Skipping — only {len(mini_chunks)} chunk(s).")
        return []

    tagged   = _format_tagged_chunks(mini_chunks)
    user_msg = _USER_TEMPLATE.format(n=len(mini_chunks), tagged=tagged)
    n        = len(mini_chunks)

    for backend in backends:  # chạy danh sách  llm sử dụng
        print(f"    [LLM-SYNC] Trying: {backend.label}")
        for attempt in range(1, max_retries + 1): # Nếu llm nào lỗi thì thử lại sau 2*số lần (tối đa 3 lần )
            raw = ""
            try:
                raw   = backend.call(_SYSTEM_PROMPT, user_msg)
                print(f"llm trả lời \n {raw} \n\n\n" )
                valid = _parse_split_response(raw, n)
                print(f"    [LLM-SYNC/{backend.label}] attempt={attempt} | "
                      f"chunks={n} → splits={valid}")
                return valid
            except json.JSONDecodeError as exc:
                print(f"    [LLM-SYNC/{backend.label}] JSON error "
                      f"(attempt {attempt}/{max_retries}): {exc} | "
                      f"raw={repr(raw[:120])}")
            except Exception as exc:
                print(f"    [LLM-SYNC/{backend.label}] Error "
                      f"(attempt {attempt}/{max_retries}): "
                      f"{type(exc).__name__}: {exc}")
            if attempt < max_retries:
                wait = 2 ** attempt
                print(f"    [LLM-SYNC/{backend.label}] Waiting {wait}s…")
                time.sleep(wait)
        print(f"    [LLM-SYNC/{backend.label}] Exhausted — next backend.")

    print("    [LLM-SYNC] ALL backends failed → zero splits for this window.")
    return []


# Hàm này chạy chế độ Hàng loạt - Batch. Giúp giảm 50% chi phí cho dữ liệu lớn

class GeminiBatchOrchestrator:

    SUB_BATCH_SIZE: int = 40      # chunks per JSONL line / LLM request
    POLL_INTERVAL:  int = 30      # seconds between status polls
    POLL_TIMEOUT:   int = 7_200   # 2 hours max wait

    # Gemini batch job states
    _DONE_STATES:   frozenset[str] = frozenset({
        "JOB_STATE_SUCCEEDED", "SUCCEEDED",
    })
    _FAILED_STATES: frozenset[str] = frozenset({
        "JOB_STATE_FAILED", "FAILED",
        "JOB_STATE_CANCELLED", "CANCELLED",
    })

    def __init__(self, api_key: str, model_id: str = GEMINI_MODEL_ID):
        from google import genai
        self._client   = genai.Client(api_key=api_key)
        self._api_key  = api_key
        self._model    = model_id

    # ── Public entry point ────────────────────────────────────────────────────

    def run(
        self,
        all_window_chunks: list[list[MiniChunk]],
    ) -> dict[int, list[int]]:

        total_chunks = sum(len(c) for c in all_window_chunks)
        total_lines  = sum(
            -(-len(c) // self.SUB_BATCH_SIZE)   # ceil division
            for c in all_window_chunks if c
        )
        print(f"[BATCH] model={self._model} | "
              f"windows={len(all_window_chunks)} | "
              f"total_chunks={total_chunks:,} | "
              f"jsonl_lines={total_lines:,}")

        # ── Step 1 ──────────────────────────────────────────────────────────
        print("[BATCH] Step 1/3 — Generating JSONL input file…")
        jsonl_path, request_map = self._generate_jsonl(all_window_chunks)
        file_size_kb = pathlib.Path(jsonl_path).stat().st_size / 1024
        print(f"[BATCH] JSONL written → '{jsonl_path}' "
              f"({file_size_kb:.1f} KB, {len(request_map)} lines)")

        # ── Step 2 ──────────────────────────────────────────────────────────
        print("[BATCH] Step 2/3 — Uploading file and creating batch job…")
        job = self._upload_and_create_job(jsonl_path)
        print(f"[BATCH] Job created → name='{job.name}' | "
              f"state={getattr(job, 'state', 'PENDING')}")

        # ── Step 3 ──────────────────────────────────────────────────────────
        print("[BATCH] Step 3/3 — Polling job status…")
        all_splits = self._poll_and_collect(job, request_map)
        windows_with_splits = sum(1 for v in all_splits.values() if v)
        print(f"[BATCH] Done — "
              f"{windows_with_splits}/{len(all_window_chunks)} windows have splits | "
              f"total split points={sum(len(v) for v in all_splits.values())}")
        return all_splits

    # ── Step 1: Generate JSONL ────────────────────────────────────────────────

    def _generate_jsonl(
        self,
        all_window_chunks: list[list[MiniChunk]],
    ) -> tuple[str, dict[str, tuple[int, int]]]:

        jsonl_path  = "batch_input.jsonl"
        request_map: dict[str, tuple[int, int]] = {}

        # Pre-compute the JSON schema dict once (Pydantic v2 API)
        schema_dict = SplitResponse.model_json_schema()

        lines_written = 0
        with open(jsonl_path, "w", encoding="utf-8") as fh:
            for wi, chunks in enumerate(all_window_chunks):
                if not chunks:
                    continue

                # Split chunks into sub-batches of SUB_BATCH_SIZE
                for start in range(0, len(chunks), self.SUB_BATCH_SIZE):
                    sub_batch = chunks[start: start + self.SUB_BATCH_SIZE]
                    if not sub_batch:
                        continue

                    n         = len(sub_batch)
                    custom_id = f"w{wi}_off{start}"

                    # Re-index tags 0..N-1 inside each sub-batch so the model
                    # always receives 0-based indices regardless of mc.idx.
                    tagged   = _format_tagged_chunks_zero_indexed(sub_batch)
                    user_msg = _USER_TEMPLATE.format(n=n, tagged=tagged)

                    line_obj = {
                        "custom_id": custom_id,
                        "request": {
                            "contents": [
                                {
                                    "role": "user",
                                    "parts": [{"text": user_msg}]
                                }
                            ],
                            # Sử dụng snake_case đồng bộ theo chuẩn SDK
                            "generation_config": {
                                "temperature": 0.0,
                                "response_mime_type": "application/json",
                                "response_schema": schema_dict
                            },
                            "system_instruction": {
                                "parts": [{"text": _SYSTEM_PROMPT}]
                            }
                        }
                    }
                    fh.write(json.dumps(line_obj, ensure_ascii=False) + "\n")
                    request_map[custom_id] = (wi, start)
                    lines_written += 1

        print(f"[BATCH] Generated {lines_written} JSONL lines "
              f"covering {len(all_window_chunks)} windows.")
        return jsonl_path, request_map

    # ── Step 2: Upload + create job ───────────────────────────────────────────

    def _upload_and_create_job(self, jsonl_path: str):
        """
        Upload the JSONL file via the Google Files API and submit a Batch job.
        Returns the created job object.
        """
        # 1. Import module types của thư viện google genai để lấy Object cấu hình
        from google.genai import types as genai_types

        print(f"[BATCH] Uploading '{jsonl_path}' via Files API…")

        # 2. Định nghĩa cấu hình mime_type thông qua UploadFileConfig của SDK
        upload_config = genai_types.UploadFileConfig(mime_type="text/plain")

        # 3. Gọi hàm upload và truyền object cấu hình vào tham số config
        uploaded_file = self._client.files.upload(
            file=jsonl_path,
            config=upload_config
        )
        print(f"[BATCH] File uploaded → name='{uploaded_file.name}'")

        print(f"[BATCH] Creating batch job (model={self._model})…")
        job = self._client.batches.create(
            model=self._model,
            src=uploaded_file.name,
        )
        return job

    # ── Step 3: Poll, download, remap ─────────────────────────────────────────

    def _poll_and_collect(
        self,
        job,
        request_map: dict[str, tuple[int, int]],
    ) -> dict[int, list[int]]:

        deadline    = time.time() + self.POLL_TIMEOUT
        elapsed     = 0
        last_state  = ""

        # ── Polling loop ──────────────────────────────────────────────────────
        while True:
            job_status  = self._client.batches.get(name=job.name)
            state_name  = str(getattr(job_status, "state", "UNKNOWN"))

            # Normalise: some SDK versions return enum repr like "BatchJobState.SUCCEEDED"
            state_upper = state_name.upper().split(".")[-1]

            if state_upper != last_state:
                print(f"[BATCH] Job state: {state_upper} "
                      f"(elapsed {elapsed}s)")
                last_state = state_upper

            if state_upper in {s.split("_", 1)[-1] for s in self._DONE_STATES} \
                    or state_upper in self._DONE_STATES:
                print(f"[BATCH] Job SUCCEEDED after {elapsed}s.")
                break

            if state_upper in {s.split("_", 1)[-1] for s in self._FAILED_STATES} \
                    or state_upper in self._FAILED_STATES:
                print("\n" + "="*40)
                print("[GEMINI BATCH ERROR DIAGNOSIS]")
                print(f"Error Details: {getattr(job_status, 'error', 'No detailed error field found')}")
                print("="*40 + "\n")
                raise RuntimeError(
                    f"Batch job '{job.name}' reached terminal failure state: "
                    f"{state_name}. Check Google Cloud console for details."
                )


            if time.time() > deadline:
                raise TimeoutError(
                    f"Batch job '{job.name}' did not finish within "
                    f"{self.POLL_TIMEOUT}s. "
                    f"Last state: {state_name}."
                )

            time.sleep(self.POLL_INTERVAL)
            elapsed += self.POLL_INTERVAL

        # ── Locate and download output JSONL ──────────────────────────────────
        output_content = self._download_output(job_status)

        # ── Parse output and remap indices ────────────────────────────────────
        all_splits: dict[int, list[int]] = {}
        parsed_ok  = 0
        parsed_err = 0

        for line_no, line in enumerate(output_content.splitlines(), start=1):
            line = line.strip()
            if not line:
                continue

            try:
                record    = json.loads(line)
                custom_id = record.get("custom_id", "")

                # Check for an API-level error on this specific request
                if record.get("error"):
                    err_msg = record["error"].get("message", "unknown error")
                    print(f"[BATCH] Line {line_no}: custom_id='{custom_id}' "
                          f"API error → {err_msg} (skipping)")
                    parsed_err += 1
                    continue

                # Decode tracking id → (wi, sub_batch_start)
                if custom_id not in request_map:
                    print(f"[BATCH] Line {line_no}: unknown custom_id "
                          f"'{custom_id}' (skipping)")
                    parsed_err += 1
                    continue

                wi, start = request_map[custom_id]

                # Extract text from the first candidate
                response   = record.get("response", {}) or {}
                candidates = response.get("candidates", [])
                if not candidates:
                    print(f"[BATCH] Line {line_no}: no candidates for "
                          f"'{custom_id}' (skipping)")
                    parsed_err += 1
                    continue

                parts = (candidates[0]
                         .get("content", {})
                         .get("parts", []))
                raw_text = parts[0].get("text", "") if parts else ""

                # Determine sub-batch size for index validation
                # = number of sub-batch entries that were sent for this custom_id
                # We reconstruct it from how many chunks are in this window
                # (we don't store it, so use SUB_BATCH_SIZE as upper bound)
                local_indices = _parse_split_response(raw_text, self.SUB_BATCH_SIZE)

                # Remap: local 0-based → absolute mc.idx within the window
                absolute = [start + li for li in local_indices]

                if wi not in all_splits:
                    all_splits[wi] = []
                all_splits[wi].extend(absolute)
                parsed_ok += 1

            except Exception as exc:
                print(f"[BATCH] Line {line_no}: parse error — "
                      f"{type(exc).__name__}: {exc} (skipping)")
                parsed_err += 1

        # Deduplicate and sort each window's split list
        for wi in all_splits:
            all_splits[wi] = sorted(set(all_splits[wi]))

        print(f"[BATCH] Output parsed — "
              f"ok={parsed_ok} | errors/skipped={parsed_err} | "
              f"windows_with_splits={len(all_splits)}")
        return all_splits

    # ── Output file download ──────────────────────────────────────────────────

    def _download_output(self, job_status) -> str:
        """
        Locate and download the output JSONL produced by the batch job.

        The output file reference is in job_status.dest.
        We resolve the Files API name, then download the raw bytes via
        the Google REST download endpoint authenticated with the API key.

        Falls back to writing a local 'batch_output.jsonl' if a download
        URL can be determined.
        """
        # ── Locate output file name ────────────────────────────────────────
        dest = getattr(job_status, "dest", None)
        if dest is None:
            raise RuntimeError(
                "job_status.dest is None — cannot locate output file. "
                "Check that the batch job used the Files API source."
            )

        # dest may be a BatchJobDestination object with a file_name attr,
        # or a plain string, depending on SDK version.
        if hasattr(dest, "file_name"):
            output_name: str = dest.file_name        # e.g. "files/abc123"
        elif isinstance(dest, str):
            output_name = dest
        else:
            # Try common attribute names for different SDK versions
            for attr in ("file_name", "name", "uri"):
                val = getattr(dest, attr, None)
                if val:
                    output_name = str(val)
                    break
            else:
                raise RuntimeError(
                    f"Cannot determine output file name from dest={dest!r}. "
                    "Inspect job_status.dest manually and adjust _download_output()."
                )

        print(f"[BATCH] Output file name: '{output_name}'")

        # ── Build download URL ─────────────────────────────────────────────
        # Files API download endpoint:
        # GET /download/v1beta/{name}:download?alt=media
        # Auth: x-goog-api-key header
        download_url = (
            "https://generativelanguage.googleapis.com/download/v1beta/"
            f"{output_name}:download?alt=media"
        )

        print(f"[BATCH] Downloading output JSONL…")
        req = urllib.request.Request(
            download_url,
            headers={"x-goog-api-key": self._api_key},
        )
        try:
            with urllib.request.urlopen(req) as resp:
                content = resp.read().decode("utf-8")
        except urllib.error.HTTPError as exc:
            raise RuntimeError(
                f"Failed to download batch output from '{download_url}': "
                f"HTTP {exc.code} {exc.reason}. "
                "Verify the API key and that the file has not expired."
            ) from exc

        # Save locally for inspection / re-use
        local_out = "batch_output.jsonl"
        with open(local_out, "w", encoding="utf-8") as fh:
            fh.write(content)

        lines = [l for l in content.splitlines() if l.strip()]
        print(f"[BATCH] Output downloaded → '{local_out}' "
              f"({len(content):,} chars, {len(lines)} non-empty lines)")
        return content


### Semantic Merging
Merge theo kết quả của gemini trả về  
Và overlap các chunk

In [758]:


def _tok_count_full(text: str, tokenizer) -> int:
    return len(tokenizer.encode(text, add_special_tokens=True, truncation=False))

 # group theo danh sách mà llm gợi ý
def _group_by_splits(
    chunks: list[MiniChunk], split_indices: list[int]
) -> list[list[MiniChunk]]:

    if not chunks:
        return []
    split_set = set(split_indices)
    groups:  list[list[MiniChunk]] = []
    current: list[MiniChunk]       = []
    # thêm các chunks vào current. Nếu gặp điểm cắt thì đẩy vào groups rồi làm sạch
    for mc in chunks:
        current.append(mc)
        if mc.idx in split_set:
            groups.append(current)
            current = []
    if current:
        groups.append(current)
    return groups


  #  raw_text   — văn bản gốc
  # embed_input— raw_text + a semantic overlap slice
  # groups chứa list chunks. trong chunks có các minichunks
def build_chunk_texts(
    groups: list[list[MiniChunk]], tokenizer
) -> list[tuple[str, str]]:

    results: list[tuple[str, str]] = []

    for i, group in enumerate(groups):
        raw   = "".join(mc.text for mc in group)
        embed = raw   # default: no overlap

        if i + 1 < len(groups): # lấy văn bản của chunk tiếp theo
            next_raw  = "".join(mc.text for mc in groups[i + 1])
            next_toks = tokenizer.encode(next_raw, add_special_tokens=False,
                                         truncation=False)
            n_next = len(next_toks)

            if n_next > 0:
                for ovl in range(OVERLAP_TOK_MAX, OVERLAP_TOK_MIN - 1, -1):
                    # Estimate character count corresponding to `ovl` tokens
                    # using the token/char ratio of next_raw.
                    ratio    = min(ovl / n_next, 1.0)
                    char_end = max(1, int(len(next_raw) * ratio))

                    # Snap forward to the next whitespace to avoid mid-word cut
                    while (char_end < len(next_raw) and
                           next_raw[char_end] not in (" ", "\n", "\t", "\r")):
                        char_end += 1

                    ovl_slice = next_raw[:char_end]   # pure source slice
                    candidate = raw + " " + ovl_slice

                    if _tok_count_full(candidate, tokenizer) <= MAX_EMBED_TOKENS:
                        embed = candidate
                        break
                    # If MIN overlap still overflows, leave embed = raw
        embed = re.sub((r"\[(\d{1,2})\]"), ' ', embed)
        results.append((raw, embed))

    return results


 # gắn footer
def collect_group_metadata(
    groups: list[list[MiniChunk]], page_footnotes_map: dict
) -> list[dict]:
    metas: list[dict] = []

    for group in groups:
        all_pages: list[int] = []
        all_refs:  list[str] = []
        for mc in group:
            all_pages.extend(mc.pages)
            all_refs.extend(mc.footnote_refs)

        pages = sorted(set(all_pages))
        refs  = list(dict.fromkeys(all_refs))   # ordered dedup

        footnotes: dict[str, str] = {}
        for ref in refs:
            for pg in pages:
                pg_str = str(pg)
                if (pg_str in page_footnotes_map and
                        ref in page_footnotes_map[pg_str]):
                    footnotes[ref] = page_footnotes_map[pg_str][ref]
                    break   # first page defining this ref wins

        metas.append({
            "pages":          pages,
            "footnote_refs":  refs,
            "footnotes":      footnotes,
        })

    return metas



### Word Segmentation (underthesea) and tokenizer

In [759]:


def segment_underthesea(text: str) -> str:
    return word_tokenize(text, format="text")


def inspect_tokens(
    raw: str, seg: str, tokenizer, chunk_id: str
) -> int:
    ids   = tokenizer.encode(
        seg,
        add_special_tokens=True,
        truncation=True,
        max_length=MAX_EMBED_TOKENS,
    )
    count = len(ids)

    print(f"      [TOKEN] id={chunk_id[:8]}… | "
          f"raw[0:60]={raw[:60].strip()!r}")
    print(f"              seg[0:60]={seg[:60].strip()!r}")
    print(f"              ids[:8]={ids[:8]} | total_tokens={count}")
    return count



### Embedding Model

In [760]:

def _mean_pool(model_output, attention_mask: torch.Tensor) -> torch.Tensor:
    """Compute attention-mask-weighted mean pool over token embeddings."""
    tok_emb  = model_output.last_hidden_state            # (B, T, D)
    mask_exp = attention_mask.unsqueeze(-1).expand(tok_emb.size()).float()
    return torch.sum(tok_emb * mask_exp, dim=1) / mask_exp.sum(dim=1).clamp(min=1e-9)


def embed_text(
    seg_text: str,
    tokenizer,
    model:  AutoModel,
    device: str,
) -> list[float]:
    enc = tokenizer(
        seg_text,
        padding=True,
        truncation=True,
        max_length=MAX_EMBED_TOKENS,
        return_tensors="pt",
    ).to(device)

    with torch.no_grad():
        out = model(**enc)

    emb = _mean_pool(out, enc["attention_mask"])
    emb = torch.nn.functional.normalize(emb, p=2, dim=1)
    return emb[0].cpu().numpy().tolist()


###  QDRANT PAYLOAD

$$\text{Chuỗi text thô} \xrightarrow{\text{Underthesea}} \text{word segmented} \xrightarrow{\text{Encoder + Pooling}} \text{Vector [768 chiều]} \xrightarrow{\text{Đóng gói}} \text{Qdrant Database}$$

In [761]:

def to_qdrant_point(chunk: FinalChunk) -> dict:
    return {
        "id":      chunk.chunk_id,
        # "vector":  chunk.vector,
        "payload": {
            "book_name":      chunk.book_name,
            "pages":          chunk.pages,
            "raw_text":       chunk.raw_text,
            "segmented_text": chunk.segmented_text,
            "footnote_refs":  chunk.footnote_refs,
            "footnotes":      chunk.footnotes,
            "token_count":    chunk.token_count,
        },
    }



### MAIN

In [762]:
def run_pipeline(
    input_json:         dict,
    backends:           list[LLMBackend],
    batch_orchestrator: Optional[GeminiBatchOrchestrator] = None,
) -> list[dict]:

    book_name          = input_json["book_name"]
    full_text          = input_json["full_text"]
    page_footnotes_map = input_json["page_footnotes_map"]

    mode = "BATCH (Mode A)" if batch_orchestrator else "SYNC (Mode B)"
    if batch_orchestrator:
        llm_label = f"GeminiBatch/{batch_orchestrator._model}"
    else:
        llm_label = " → ".join(b.label for b in backends)

    print("═" * 72)
    print(f"  PIPELINE START — '{book_name}'")
    print(f"  full_text chars     : {len(full_text):>12,}")
    print(f"  footnote page count : {len(page_footnotes_map):>12,}")
    print(f"  LLM mode            : {mode}")
    print(f"  LLM backend(s)      : {llm_label}")
    print("═" * 72)

    print("═" * 72)
    print(f"  PIPELINE START — '{book_name}'")
    print(f"  full_text chars     : {len(full_text):>12,}")
    print(f"  footnote page count : {len(page_footnotes_map):>12,}")
    print("═" * 72)

 # gọi gemini
    # if gemini_api_key:
    #     gemini_client = genai.Client(api_key=gemini_api_key)
    # else:
    #     gemini_client = genai.Client()   # reads GOOGLE_API_KEY env var

    # provider_labels = " → ".join(b.label for b in backends)


    print(f"[INIT] Loading tokenizer: {EMBED_MODEL_ID}")
    tokenizer  = AutoTokenizer.from_pretrained(EMBED_MODEL_ID)

    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"[INIT] Loading model → device={device}")
    embed_model = AutoModel.from_pretrained(EMBED_MODEL_ID).to(device).eval()
    print(f"[INIT] Model ready | hidden_size={embed_model.config.hidden_size}")

# bóc tách trang
    spans = parse_page_spans(full_text)

# chia sách thành các trang cho các lần gọi api
    windows = create_dynamic_can_chi_windows(spans, book_name)

    all_points:    list[dict] = []
    total_windows: int        = len(windows)

    print("\n── STEPS 3+4 · Structural boundaries + Mini-chunking (all windows)")
    all_mc: list[list[MiniChunk]] = []
    for wi, window in enumerate(windows):
        pg_str = str(window.pages[:4]) + ("…" if len(window.pages) > 4 else "")
        print(f"  [PREP] window {wi + 1:>4}/{total_windows} | pages={pg_str}")
        all_mc.append(create_mini_chunks(window, spans, book_name))

    total_mc = sum(len(c) for c in all_mc)
    print(f"[PREP] Total mini-chunks across all windows: {total_mc:,}")

    # ── Step 5: LLM semantic split ────────────────────────────────────────────
    # BATCH MODE: submit one job now; get {wi: splits} map back.
    # SYNC MODE:  per-window calls happen inside the loop below.
    batch_splits: dict[int, list[int]] = {}

    if batch_orchestrator is not None:
        print("\n── STEP 5 · Gemini Batch API ────────────────────────────────────")
        batch_splits = batch_orchestrator.run(all_mc)
        print(f"[STEP5] Batch splits received for "
              f"{len(batch_splits)}/{total_windows} windows.")

# chia từng minichunk cho danh sách trang trên
    for wi, window in enumerate(windows):

        print(f"\n{'─' * 72}")
        pg_str = str(window.pages[:6]) + ("…" if len(window.pages) > 6 else "")
        print(f"  WINDOW {wi + 1:>4}/{total_windows} | pages={pg_str} | chars={len(window.text):,}")


        mini_chunks = create_mini_chunks(window, spans, book_name)
        if not mini_chunks:
            print("    [SKIP] No content mini-chunks produced for this window.")
            continue

# gọi llm để chia chunks
        # split_indices = gemini_semantic_split(mini_chunks, backends)
        if batch_orchestrator is not None:
            split_indices = batch_splits.get(wi, [])
        else:
            print(f"    [STEP5-SYNC] Calling LLM chain…")
            split_indices = llm_semantic_split(mini_chunks, backends)
        print(f"llm trả về \n {split_indices} \n\n\n")

# Merge groups + overlap + footnote metadata
        groups       = _group_by_splits(mini_chunks, split_indices)
        chunk_texts  = build_chunk_texts(groups, tokenizer)    # (raw, embed_input)
        metas        = collect_group_metadata(groups, page_footnotes_map)

        print(f"    [MERGE] {len(mini_chunks)} mini-chunks + {len(split_indices)} splits "
              f"→ {len(groups)} semantic groups")

        points_to_upload = []
# Segment, inspect, embed, format
        for gi, ((raw_text, embed_input), meta) in enumerate(zip(chunk_texts, metas)):
            if not raw_text.strip():
                continue

            chunk_id = str(uuid6.uuid7())

      # Word segmentation (applied to embed_input which includes overlap)
            seg_text = segment_underthesea(embed_input)

      #  Token inspection + count
            tok_count = inspect_tokens(raw_text, seg_text, tokenizer, chunk_id)

      #  Dense embedding
            vector = embed_text(seg_text, tokenizer, embed_model, device)

            if hasattr(vector, "tolist"):
                    vector = vector.tolist()
            elif isinstance(vector, np.ndarray): # Nếu là mảng numpy
                vector = vector.astype(float).tolist()

      #  Assemble FinalChunk
            fc = FinalChunk(
                chunk_id       = chunk_id,
                book_name      = book_name,
                pages          = meta["pages"],
                raw_text       = raw_text,       # pure source, no mutation
                segmented_text = seg_text,        # underthesea-segmented embed input
                footnote_refs  = meta["footnote_refs"],
                footnotes      = meta["footnotes"],
                token_count    = tok_count,
                # vector         = vector,
            )

            point = to_qdrant_point(fc)
            all_points.append(point)


    # 5. Đóng gói trực tiếp vào cấu trúc PointStruct của Qdrant (Thay thế cho FinalChunk)
            point = PointStruct(
                id=chunk_id,
                vector=vector,
                payload={
                    "book_name": book_name,
                    "pages": meta["pages"],
                    "raw_text": raw_text,
                    "segmented_text": seg_text,
                    "footnote_refs": meta["footnote_refs"],
                    "footnotes": meta["footnotes"],
                    "token_count": tok_count
                }
            )
            points_to_upload.append(point)
        time.sleep(2.0)
    # 6. Đẩy toàn bộ dữ liệu lên Qdrant Cloud sau khi kết thúc vòng lặp
        if points_to_upload:
            print(f"Đang đẩy {len(points_to_upload)} points lên Qdrant Cloud...")

            # Sử dụng upload_points (hoặc upsert) để đẩy data lên Cloud
            client.upload_points(
                collection_name=COLLECTION_NAME,
                points=points_to_upload,
                batch_size=64, # Chia nhỏ gói dữ liệu gửi đi để tránh nghẽn mạng trên Colab
                wait=True
            )
            print(" Đã lưu dữ liệu thành công lên Qdrant Cloud!")
        else:
            print("Không có dữ liệu hợp lệ để đẩy.")

    print("\n" + "═" * 72)
    print(f"  PIPELINE COMPLETE — '{book_name}'")
    print(f"  Total Qdrant points produced : {len(all_points):,}")
    print("═" * 72)
    return all_points

#  RUN Đại Việt Sử Ký

In [746]:
# @title

import os
from google.colab import userdata


JSON_PATH = "full_DVSK_data.json"           # Đại Việt Sử Ký Toàn Thư
# JSON_PATH = "full_KhamDinh_data.json"        # Khâm Định Việt Sử Thông Giám Cương Mục
# JSON_PATH = "full_VietSu_data.json"                 # Việt Sử Toàn Thư
# JSON_PATH = "full_VTT_data.json"                  # Vương Triều Trần
# JSON_PATH = "full_test.json"


OUTPUT_PATH = JSON_PATH.replace(".json", "_qdrant_points.json")


def _load_key(colab_secret: str, env_var: str) -> str:
    try:
        from google.colab import userdata as _ud
        val = _ud.get(colab_secret)
        if val:
            print(f"[KEY] '{colab_secret}' loaded from Colab Secrets.")
            return val
    except Exception:
        pass
    val = os.environ.get(env_var, "")
    if val:
        print(f"[KEY] '{env_var}' loaded from environment variable.")
    return val

GROQ_API_KEY   = _load_key("grok_key",  "grok_key")
GEMINI_API_KEY = _load_key("GEMINI_API_KEY", "GEMINI_API_KEY")

if not GROQ_API_KEY and not GEMINI_API_KEY:
    print("[KEY] ⚠  No API keys found. "
          "Add GROQ_API_KEY and/or GOOGLE_API_KEY to Colab Secrets.")


# USE_BATCH_MODE: bool = True
USE_BATCH_MODE: bool = False

BATCH_ORCHESTRATOR: Optional[GeminiBatchOrchestrator] = None
LLM_BACKENDS:       list[LLMBackend]                  = []

if USE_BATCH_MODE:
    if not GEMINI_API_KEY:
        raise RuntimeError(
            "USE_BATCH_MODE=True but GOOGLE_API_KEY is not set. "
            "Add it to Colab Secrets or switch USE_BATCH_MODE=False."
        )
    BATCH_ORCHESTRATOR = GeminiBatchOrchestrator(
        api_key=GEMINI_API_KEY,
        model_id=GEMINI_MODEL_ID,       # "gemini-2.5-flash"
    )
    print(f"[INIT] Mode A — GeminiBatchOrchestrator ready "
          f"(model={GEMINI_MODEL_ID}, "
          f"sub_batch={GeminiBatchOrchestrator.SUB_BATCH_SIZE} chunks/line)")

else:
    # Build sync fallback chain
    # if GROQ_API_KEY:
    #     LLM_BACKENDS.append(GroqBackend(GROQ_API_KEY, GROQ_MODEL_LLAMA_70B))
    #     LLM_BACKENDS.append(GroqBackend(GROQ_API_KEY, GROQ_MODEL_LLAMA_8B))
    #     LLM_BACKENDS.append(GroqBackend(GROQ_API_KEY, GROQ_MODEL_QWEN))

    if GEMINI_API_KEY:
        try:
            LLM_BACKENDS.append(GeminiBackend(GEMINI_API_KEY, GEMINI_MODEL_ID))
            print("[INIT] GeminiBackend added as last-resort sync fallback.")
        except Exception as _ge:
            print(f"[INIT] GeminiBackend skipped ({type(_ge).__name__}: {_ge}).")

    if not LLM_BACKENDS:
        raise RuntimeError(
            "USE_BATCH_MODE=False but LLM_BACKENDS is empty. "
            "Provide GROQ_API_KEY and/or GOOGLE_API_KEY."
        )

    print(f"[INIT] Mode B — sync fallback chain ({len(LLM_BACKENDS)} backend(s)):")
    for _i, _b in enumerate(LLM_BACKENDS):
        print(f"   [{_i + 1}] {_b.label}")

print()

print(f"[LOAD] Reading '{JSON_PATH}'…")
with open(JSON_PATH, "r", encoding="utf-8") as _fh:
    input_data: dict = json.load(_fh)


assert "book_name"          in input_data, "Missing key: book_name"
assert "full_text"          in input_data, "Missing key: full_text"
assert "page_footnotes_map" in input_data, "Missing key: page_footnotes_map"
assert isinstance(input_data["full_text"],          str),  "full_text must be str"
assert isinstance(input_data["page_footnotes_map"], dict), "page_footnotes_map must be dict"
print("[LOAD] Input validation passed ✓")

qdrant_points: list[dict] = run_pipeline(
    input_data,
    backends           = LLM_BACKENDS,
    batch_orchestrator = BATCH_ORCHESTRATOR,
)

print(f"\n[SAVE] Writing {len(qdrant_points):,} points → '{OUTPUT_PATH}'…")
with open(OUTPUT_PATH, "w", encoding="utf-8") as _fh:
    json.dump(qdrant_points, _fh, ensure_ascii=False, indent=2)
_kb = os.path.getsize(OUTPUT_PATH) / 1024
print(f"[SAVE] Done. {_kb:.1f} KB written.")


[KEY] 'grok_key' loaded from Colab Secrets.
[KEY] 'GEMINI_API_KEY' loaded from Colab Secrets.
[INIT] GeminiBackend added as last-resort sync fallback.
[INIT] Mode B — sync fallback chain (1 backend(s)):
   [1] gemini-sync/gemini-3.1-flash-lite

[LOAD] Reading 'full_DVSK_data.json'…
[LOAD] Input validation passed ✓
════════════════════════════════════════════════════════════════════════
  PIPELINE START — 'Đại Việt Sử Ký Toàn Thư'
  full_text chars     :      403,175
  footnote page count :          154
  LLM mode            : SYNC (Mode B)
  LLM backend(s)      : gemini-sync/gemini-3.1-flash-lite
════════════════════════════════════════════════════════════════════════
════════════════════════════════════════════════════════════════════════
  PIPELINE START — 'Đại Việt Sử Ký Toàn Thư'
  full_text chars     :      403,175
  footnote page count :          154
════════════════════════════════════════════════════════════════════════
[INIT] Loading tokenizer: bkai-foundation-models/vietnames

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[INIT] Model ready | hidden_size=768

[WINDOWS] book='Đại Việt Sử Ký Toàn Thư' | window_size=5 pages

── STEPS 3+4 · Structural boundaries + Mini-chunking (all windows)
  [PREP] window    1/31 | pages=[154, 155, 156, 157]…
      [STRUCT] 13 structural boundaries | book='Đại Việt Sử Ký Toàn Thư'
    [MINI-CHUNK] window chars=14,459 → 132 mini-chunks
  [PREP] window    2/31 | pages=[158, 159, 160, 161]…
      [STRUCT] 8 structural boundaries | book='Đại Việt Sử Ký Toàn Thư'
    [MINI-CHUNK] window chars=13,036 → 119 mini-chunks
  [PREP] window    3/31 | pages=[163, 164, 165, 166]…
      [STRUCT] 13 structural boundaries | book='Đại Việt Sử Ký Toàn Thư'
    [MINI-CHUNK] window chars=12,645 → 113 mini-chunks
  [PREP] window    4/31 | pages=[168, 169, 170, 171]…
      [STRUCT] 11 structural boundaries | book='Đại Việt Sử Ký Toàn Thư'
    [MINI-CHUNK] window chars=11,532 → 102 mini-chunks
  [PREP] window    5/31 | pages=[173, 174, 175, 176]…
      [STRUCT] 7 structural boundaries | book='Đại

# Chạy Khâm Định Việt Sử Thông Giám Cương Mục

In [ ]:
# @title

import os
from google.colab import userdata


# JSON_PATH = "full_DVSK_data.json"           # Đại Việt Sử Ký Toàn Thư
JSON_PATH = "full_KhamDinh_data.json"        # Khâm Định Việt Sử Thông Giám Cương Mục
# JSON_PATH = "full_VietSu_data.json"                 # Việt Sử Toàn Thư
# JSON_PATH = "full_VTT_data.json"                  # Vương Triều Trần
# JSON_PATH = "full_test.json"


OUTPUT_PATH = JSON_PATH.replace(".json", "_qdrant_points.json")


def _load_key(colab_secret: str, env_var: str) -> str:
    try:
        from google.colab import userdata as _ud
        val = _ud.get(colab_secret)
        if val:
            print(f"[KEY] '{colab_secret}' loaded from Colab Secrets.")
            return val
    except Exception:
        pass
    val = os.environ.get(env_var, "")
    if val:
        print(f"[KEY] '{env_var}' loaded from environment variable.")
    return val

GROQ_API_KEY   = _load_key("grok_key",  "grok_key")
GEMINI_API_KEY = _load_key("GEMINI_API_KEY", "GEMINI_API_KEY")

if not GROQ_API_KEY and not GEMINI_API_KEY:
    print("[KEY] ⚠  No API keys found. "
          "Add GROQ_API_KEY and/or GOOGLE_API_KEY to Colab Secrets.")


# USE_BATCH_MODE: bool = True
USE_BATCH_MODE: bool = False

BATCH_ORCHESTRATOR: Optional[GeminiBatchOrchestrator] = None
LLM_BACKENDS:       list[LLMBackend]                  = []

if USE_BATCH_MODE:
    if not GEMINI_API_KEY:
        raise RuntimeError(
            "USE_BATCH_MODE=True but GOOGLE_API_KEY is not set. "
            "Add it to Colab Secrets or switch USE_BATCH_MODE=False."
        )
    BATCH_ORCHESTRATOR = GeminiBatchOrchestrator(
        api_key=GEMINI_API_KEY,
        model_id=GEMINI_MODEL_ID,       # "gemini-2.5-flash"
    )
    print(f"[INIT] Mode A — GeminiBatchOrchestrator ready "
          f"(model={GEMINI_MODEL_ID}, "
          f"sub_batch={GeminiBatchOrchestrator.SUB_BATCH_SIZE} chunks/line)")

else:
    # Build sync fallback chain
    # if GROQ_API_KEY:
    #     LLM_BACKENDS.append(GroqBackend(GROQ_API_KEY, GROQ_MODEL_LLAMA_70B))
    #     LLM_BACKENDS.append(GroqBackend(GROQ_API_KEY, GROQ_MODEL_LLAMA_8B))
    #     LLM_BACKENDS.append(GroqBackend(GROQ_API_KEY, GROQ_MODEL_QWEN))

    if GEMINI_API_KEY:
        try:
            LLM_BACKENDS.append(GeminiBackend(GEMINI_API_KEY, GEMINI_MODEL_ID))
            print("[INIT] GeminiBackend added as last-resort sync fallback.")
        except Exception as _ge:
            print(f"[INIT] GeminiBackend skipped ({type(_ge).__name__}: {_ge}).")

    if not LLM_BACKENDS:
        raise RuntimeError(
            "USE_BATCH_MODE=False but LLM_BACKENDS is empty. "
            "Provide GROQ_API_KEY and/or GOOGLE_API_KEY."
        )

    print(f"[INIT] Mode B — sync fallback chain ({len(LLM_BACKENDS)} backend(s)):")
    for _i, _b in enumerate(LLM_BACKENDS):
        print(f"   [{_i + 1}] {_b.label}")

print()

print(f"[LOAD] Reading '{JSON_PATH}'…")
with open(JSON_PATH, "r", encoding="utf-8") as _fh:
    input_data: dict = json.load(_fh)


assert "book_name"          in input_data, "Missing key: book_name"
assert "full_text"          in input_data, "Missing key: full_text"
assert "page_footnotes_map" in input_data, "Missing key: page_footnotes_map"
assert isinstance(input_data["full_text"],          str),  "full_text must be str"
assert isinstance(input_data["page_footnotes_map"], dict), "page_footnotes_map must be dict"
print("[LOAD] Input validation passed ✓")

qdrant_points: list[dict] = run_pipeline(
    input_data,
    backends           = LLM_BACKENDS,
    batch_orchestrator = BATCH_ORCHESTRATOR,
)

print(f"\n[SAVE] Writing {len(qdrant_points):,} points → '{OUTPUT_PATH}'…")
with open(OUTPUT_PATH, "w", encoding="utf-8") as _fh:
    json.dump(qdrant_points, _fh, ensure_ascii=False, indent=2)
_kb = os.path.getsize(OUTPUT_PATH) / 1024
print(f"[SAVE] Done. {_kb:.1f} KB written.")


[KEY] 'grok_key' loaded from Colab Secrets.
[KEY] 'GEMINI_API_KEY' loaded from Colab Secrets.
[INIT] GeminiBackend added as last-resort sync fallback.
[INIT] Mode B — sync fallback chain (1 backend(s)):
   [1] gemini-sync/gemini-3.1-flash-lite

[LOAD] Reading 'full_KhamDinh_data.json'…
[LOAD] Input validation passed ✓
════════════════════════════════════════════════════════════════════════
  PIPELINE START — 'Khâm Định Việt Sử Thông Giám Cương Mục'
  full_text chars     :      434,220
  footnote page count :          165
  LLM mode            : SYNC (Mode B)
  LLM backend(s)      : gemini-sync/gemini-3.1-flash-lite
════════════════════════════════════════════════════════════════════════
════════════════════════════════════════════════════════════════════════
  PIPELINE START — 'Khâm Định Việt Sử Thông Giám Cương Mục'
  full_text chars     :      434,220
  footnote page count :          165
════════════════════════════════════════════════════════════════════════
[INIT] Loading tokenizer

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[INIT] Model ready | hidden_size=768

[WINDOWS] book='Khâm Định Việt Sử Thông Giám Cương Mục' | window_size=3 pages

── STEPS 3+4 · Structural boundaries + Mini-chunking (all windows)
  [PREP] window    1/53 | pages=[187, 188, 189]
      [STRUCT] 2 structural boundaries | book='Khâm Định Việt Sử Thông Giám Cương '
    [MINI-CHUNK] window chars=7,089 → 63 mini-chunks
  [PREP] window    2/53 | pages=[189, 190, 191, 192]
      [STRUCT] 2 structural boundaries | book='Khâm Định Việt Sử Thông Giám Cương '
    [MINI-CHUNK] window chars=7,783 → 66 mini-chunks
  [PREP] window    3/53 | pages=[192, 193, 194, 195]
      [STRUCT] 2 structural boundaries | book='Khâm Định Việt Sử Thông Giám Cương '
    [MINI-CHUNK] window chars=6,692 → 59 mini-chunks
  [PREP] window    4/53 | pages=[195, 196, 197, 198]
      [STRUCT] 2 structural boundaries | book='Khâm Định Việt Sử Thông Giám Cương '
    [MINI-CHUNK] window chars=8,708 → 80 mini-chunks
  [PREP] window    5/53 | pages=[198, 199, 200, 201]
      [S

# Mục mới